# European Soccer Database — Exploration

Source: [hugomathien/soccer on Kaggle](https://www.kaggle.com/datasets/hugomathien/soccer)

Run `python scripts/download_data.py` from the repo root first so that `data/database.sqlite` exists.

This notebook walks through the database structure: it lists the tables, previews a few rows of each, and shows the column schema. It's used as a starting point before building features for the outcome predictor.

In [2]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = REPO_ROOT / "data" / "database.sqlite"
assert DB_PATH.exists(), (
    f"Missing {DB_PATH}. Run `python scripts/download_data.py` from the repo root."
)

conn = sqlite3.connect(DB_PATH)
print(f"Opened {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MB)")

Opened /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite (298.6 MB)


## Tables in the database

In [3]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)
tables

,name
0,Country
1,League
2,Match
3,Player
4,Player_Attributes
5,Team
6,Team_Attributes
7,sqlite_sequence


In [4]:
# Row counts per table
row_counts = {
    name: pd.read_sql(f'SELECT COUNT(*) AS n FROM "{name}"', conn)["n"].iloc[0]
    for name in tables["name"]
}
pd.Series(row_counts, name="row_count").sort_values(ascending=False)

Player_Attributes    183978
Match                 25979
Player                11060
Team_Attributes        1458
Team                    299
Country                  11
League                   11
sqlite_sequence           7
Name: row_count, dtype: int64

## Schema and head of every table

In [5]:
for name in tables["name"]:
    print(f"\n===== {name} =====")
    schema = pd.read_sql(f'PRAGMA table_info("{name}");', conn)
    print(schema[["name", "type", "notnull", "pk"]].to_string(index=False))
    head = pd.read_sql(f'SELECT * FROM "{name}" LIMIT 3', conn)
    print(head)


===== Country =====
name    type  notnull  pk
  id INTEGER        0   1
name    TEXT        0   0
     id     name
0     1  Belgium
1  1729  England
2  4769   France

===== League =====
      name    type  notnull  pk
        id INTEGER        0   1
country_id INTEGER        0   0
      name    TEXT        0   0
     id  country_id                    name
0     1           1  Belgium Jupiler League
1  1729        1729  England Premier League
2  4769        4769          France Ligue 1

===== Match =====
            name    type  notnull  pk
              id INTEGER        0   1
      country_id INTEGER        0   0
       league_id INTEGER        0   0
          season    TEXT        0   0
           stage INTEGER        0   0
            date    TEXT        0   0
    match_api_id INTEGER        0   0
home_team_api_id INTEGER        0   0
away_team_api_id INTEGER        0   0
  home_team_goal INTEGER        0   0
  away_team_goal INTEGER        0   0
  home_player_X1 INTEGER        0 

## Quick sanity checks

A few starter queries to confirm the join keys behave as expected. Adapt these for feature engineering.

In [6]:
# Matches per league/season
pd.read_sql(
    """
    SELECT l.name AS league, m.season, COUNT(*) AS matches
    FROM Match m
    JOIN League l ON l.id = m.league_id
    GROUP BY l.name, m.season
    ORDER BY l.name, m.season;
    """,
    conn,
).head(20)

,league,season,matches
0,Belgium Jupiler League,2008/2009,306
1,Belgium Jupiler League,2009/2010,210
2,Belgium Jupiler League,2010/2011,240
3,Belgium Jupiler League,2011/2012,240
4,Belgium Jupiler League,2012/2013,240
5,Belgium Jupiler League,2013/2014,12
6,Belgium Jupiler League,2014/2015,240
7,Belgium Jupiler League,2015/2016,240
8,England Premier League,2008/2009,380
9,England Premier League,2009/2010,380


In [7]:
# Goal-difference distribution — useful as a target check for outcome prediction
pd.read_sql(
    """
    SELECT home_team_goal - away_team_goal AS goal_diff, COUNT(*) AS n
    FROM Match
    GROUP BY goal_diff
    ORDER BY goal_diff;
    """,
    conn,
)

,goal_diff,n
0,-9,1
1,-8,3
2,-7,6
3,-6,26
4,-5,85
5,-4,308
6,-3,833
7,-2,2134
8,-1,4070
9,0,6596


In [8]:
conn.close()